In [2]:
# %pip install -e ../../wazeasy
# %pip install folium mapclassify matplotlib

In [2]:
from wazeasy import utils, reports
import dask.dataframe as dd
import pandas as pd
import numpy as np
import yaml

import geopandas as gpd
from shapely import wkt

import folium
from branca.colormap import linear



In [2]:
! rm -r /dev/shm/dask-tmp

In [3]:
!mkdir /dev/shm/dask-tmp

In [4]:
import dask
dask.config.set({"temporary_directory": "/dev/shm/dask-tmp"})

In [4]:
data = gpd.read_file('../data/geography-levels/AMM_h3_level7.gpkg')
data = data.to_crs('EPSG:4326')
data.to_file('../data/AMM_h3_level7_.gpkg')

In [6]:
with open("../docs/config_montevideo.yaml", "r") as f:
    config = yaml.safe_load(f)

timezone = config["Montevideo"]["timezone"]["timezone_name"]
s3_path = config["Montevideo"]["s3_path"]
aws_profile = config["Montevideo"]["aws_profile"]
geographies = config["geographies"]

print("Timezone:", timezone)
print("S3 path:", s3_path)

Timezone: America/Montevideo
S3 path: s3://wbg-waze/bq/UY/784montevideo/jams/20*/


# Análisis longitudinal de los datos de Waze

El programa Waze for Cities Data es un intercambio gratuito y bidireccional de datos que permite a los responsables de la toma de decisiones del sector público generar un impacto concreto en sus comunidades.

Waze proporciona información sobre embotellamientos y otros eventos que afectan las condiciones de las vías. Esta información proviene de los conductores que usan Waze, conocidos como «Wazers», o de fuentes externas. Los Wazers envían reportes viales, o UGI (incidentes generados por usuarios, por sus siglas en inglés), con información detallada sobre las condiciones específicas del tráfico y de las vías. Pueden enviar reportes desde el lugar donde se encuentran o, si ya no están allí, dentro de los 30 minutos posteriores al evento. Waze también emite alertas automáticas sobre lo que denomina «tráfico inusual»: incidentes que afectan a un número excepcional de usuarios y se apartan de los patrones normales de tráfico para un día y una hora determinados.

Los datos de tráfico de Waze incluyen la siguiente información:

- Información general: marca de tiempo del archivo y área geográfica de la que se obtuvieron los datos.
- Alertas de tráfico: incidentes de tráfico reportados por los usuarios.
- Embotellamientos: información sobre la reducción de la velocidad del tráfico, generada por el servicio a partir de la ubicación y la velocidad de los usuarios.
- Tráfico inusual (irregularidades): alertas y embotellamientos que afectan a un número excepcionalmente grande de usuarios.

En este notebook utilizamos el conjunto de datos de embotellamientos (Traffic Jams).

## Conjunto de datos de embotellamientos (Jams)

La sección de embotellamientos contiene datos recopilados en tiempo real sobre reducciones de la velocidad del tráfico en segmentos viales específicos. La URL incluye un arreglo (array) de embotellamientos, junto con la fecha y la hora de inicio del reporte. Waze genera esta información a partir de las siguientes fuentes:

- Las ubicaciones GPS que envían los teléfonos de los usuarios que conducen con la aplicación abierta. Con esos puntos, Waze compara la velocidad real con la velocidad promedio para una franja horaria determinada y con la velocidad de flujo libre (la velocidad máxima registrada en el segmento vial).
- Los reportes de usuarios de Waze que encuentran embotellamientos. Estos reportes aparecen como alertas comunes y también influyen en la forma en que Waze identifica y muestra los embotellamientos.

### Limitaciones

- No sabemos cuánto varía la cantidad de usuarios de un día a otro.
- No sabemos cómo se distribuyen los usuarios en el espacio.

(muestra-embotellamientos)=
### Muestra

Utilizamos los datos de embotellamientos desde el 4 de septiembre de 2021 hasta el 30 de agosto de 2026. 

## Resultados

### Número diario de embotellamientos

El número diario de embotellamientos corresponde a la cantidad registrada cada día. También se presenta un segundo gráfico con el promedio móvil de los últimos 7 días. En el gráfico observamos lo siguiente:

- Todos los 31 de diciembre, la cantidad de embotellamientos disminuye significativamente.
- Entre julio y mediados de octubre de 2025, la cantidad de embotellamientos fluctúa de manera inesperada y registra valores bajos. Hemos observado este comportamiento en otras ciudades y lo atribuimos a cambios en el flujo de datos de Waze. Por lo tanto, no se recomienda analizar esos meses.
- Hacia finales de febrero de 2026, se observa un aumento significativo de la cantidad diaria de embotellamientos, que se mantiene durante el resto del período observado.
- Todos los años se observa una disminución de la cantidad de embotellamientos durante las fiestas, que se prolonga hasta finales de febrero o marzo.

### Número mensual de embotellamientos

En este gráfico agrupamos los embotellamientos por mes. Podemos observar lo siguiente:

- En marzo de 2026 se observa un aumento del 13,6 % respecto del mismo mes del año anterior. Este aumento se mantiene durante el resto del período observado. Como referencia, se muestra a continuación la cantidad de embotellamientos registrada en marzo de cada año:
  - Marzo de 2022: 1.473.708
  - Marzo de 2023: 1.665.475
  - Marzo de 2024: 1.585.752
  - Marzo de 2025: 1.704.405
  - Marzo de 2026: 1.937.070
- Los niveles de embotellamientos de julio, agosto, septiembre, octubre y diciembre de 2025 están por debajo de los valores habituales.

### Embotellamientos diarios por nivel

Calculamos la cantidad diaria de embotellamientos según su nivel. El nivel indica cuánto se reduce la velocidad respecto de la velocidad de flujo libre debido al embotellamiento. El nivel 1 representa la menor reducción y el nivel 2 la mayor. 

En el gráfico se observa que los embotellamientos más comunes son de nivel 2 y 3. No se aprecia ninguna anomalía, como un período en el que los embotellamientos de nivel 4 aumenten de forma desproporcionada.

### Índice de Intensidad de Congestión

Este índice busca caracterizar los embotellamientos según su duración, extensión e impacto sobre la velocidad.

```{note}
En su versión original, el índice de intensidad de la congestión (TCI, por sus siglas en inglés) resume cuánto se extienden los embotellamientos por la red vial dentro de un polígono de interés y cuánto duran. El BID lo propuso durante la pandemia de COVID-19. Waze for Cities publica un conjunto de datos de embotellamientos con registros de reducciones de velocidad en tramos viales específicos. Para cada embotellamiento, los datos incluyen la demora, la hora de inicio, la gravedad y la ubicación geográfica. Lamentablemente, no indican cuándo termina cada embotellamiento, por lo que estimamos su duración a partir de la cantidad de registros asociados a cada ID. El TCI se define entonces como la suma de todos los registros de embotellamiento dentro de un polígono durante un período determinado. Cuanto más dura un embotellamiento, más veces aparece en los datos y más contribuye al TCI.

En este trabajo, queremos caracterizar los embotellamientos no sólo por su extensión y duración, sino también por la reducción de velocidad que generan. Por esta razón, utilizamos la demora para calcular el TCI en lugar de la longitud. La demora se define como el tiempo adicional necesario para atravesar el embotellamiento y su cálculo tiene en cuenta la longitud, la velocidad de flujo libre y la velocidad durante el embotellamiento.
```

#### TCI diario

- Al representar el TCI a lo largo del período de estudio, observamos que los niveles de congestión de 2026 no son más elevados que los de años anteriores, a diferencia de lo que ocurre al comparar la cantidad de embotellamientos.
- En enero de 2026 se observa un pico en el TCI, que atribuimos a un error en los datos de base.

#### Distribución horaria del TCI

Calculamos el TCI medio para cada hora del día y mes.

- Durante la semana, la hora pico de la mañana ronda las 07:00 y la de la tarde se sitúa entre las 16:00 y las 17:00. Según el [índice de tráfico de TomTom](https://www.tomtom.com/traffic-index/city/montevideo/), las horas pico son las 08:00 y las 18:00, respectivamente.
- Durante el fin de semana, los picos ocurren unas horas más tarde: el de la mañana, a las 11:00, y el de la tarde, a las 19:00.
- Durante el verano, el TCI es menor: enero registra los valores más bajos y en febrero el TCI comienza a subir, hasta recuperar sus valores habituales en marzo. En verano, los picos ocurren más tarde: a las 10:00 por la mañana y a las 17:00 por la tarde. Los fines de semana, el pico de la tarde ocurre a las 20:00.

```{note}
Waze for Cities únicamente proporciona la hora de inicio de cada embotellamiento. Por lo tanto, no podemos determinar si se prolonga más allá de esa hora y debemos asignar todo su peso a la hora de inicio.
```

#### Comparación mensual del TCI

- Durante 2024, el TCI tiende a ser menor que en 2023 a lo largo del año. Por ejemplo, el TCI de marzo de 2024 es un 22% menor que el de marzo de 2023.
- La disminución del TCI durante el segundo semestre de 2025 se debe a un problema con los datos originales.

In [7]:
ddf = dd.read_parquet(s3_path + '*parquet', storage_options={'profile': aws_profile}, engine='pyarrow')
utils.handle_time(ddf, timezone)
utils.assign_geography_to_jams(ddf)
ddf = utils.remove_level5(ddf)

reports.run_basic_report(ddf, agg_column = 'delay')

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)